In [57]:
# import libraries
import pandas as pd

import re
from nltk.corpus import stopwords

import plotly.express as px
import plotly.graph_objects as go

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report


from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score)

from sklearn.neighbors import KNeighborsClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from scipy.sparse import vstack



### Step 4: Load and Explore Data in Python

In [37]:
# load data
train = pd.read_csv("../data/Train.csv")
test = pd.read_csv("../data/Test.csv")

# View first few rows
print(train.head())

   tweet_id                                          safe_text  label  \
0  CL1KWCMY  Me &amp; The Big Homie meanboy3000 #MEANBOY #M...    0.0   
1  E3303EME  I'm 100% thinking of devoting my career to pro...    1.0   
2  M4IVFSMS  #whatcausesautism VACCINES, DO NOT VACCINATE Y...   -1.0   
3  1DR6ROZ4  I mean if they immunize my kid with something ...   -1.0   
4  J77ENIIE  Thanks to <user> Catch me performing at La Nui...    0.0   

   agreement  
0        1.0  
1        1.0  
2        1.0  
3        1.0  
4        1.0  


In [38]:
def handle_missing_values_same_columns(train):
    # Drop rows with missing label 
    train = train.dropna(subset=['label'])
    
    # Convert label to int
    train['label'] = train['label'].astype(int)
    
    # Fill missing agreement with train median
    agreement_median = train['agreement'].median()
    train['agreement'] = train['agreement'].fillna(agreement_median)
    
    return train

In [39]:
train = handle_missing_values_same_columns(train)

In [40]:
print("Train missing values:\n", train.isnull().sum())
print("Test missing values:\n", test.isnull().sum())

Train missing values:
 tweet_id     0
safe_text    0
label        0
agreement    0
dtype: int64
Test missing values:
 tweet_id     0
safe_text    1
dtype: int64


In [41]:
# Check label distribution
print(train['label'].value_counts())

label
 0    4909
 1    4053
-1    1038
Name: count, dtype: int64


In [42]:
# visualize label distribution
# Map numeric labels to readable names
label_mapping = {
    -1: "Anti-Vaccine",
     0: "Neutral",
     1: "Pro-Vaccine"
}

# Prepare data
class_counts = train['label'].value_counts().sort_index()
total_samples = len(train['label'])

df_plot = pd.DataFrame({
    "Label": class_counts.index,
    "Count": class_counts.values
})

df_plot["Percentage"] = (df_plot["Count"] / total_samples) * 100
df_plot["Class Name"] = df_plot["Label"].map(label_mapping)

# Create interactive bar chart
fig = go.Figure()

fig.add_trace(go.Bar(
    x=df_plot["Class Name"],
    y=df_plot["Count"],
    text=[f"{c} ({p:.2f}%)" for c, p in zip(df_plot["Count"], df_plot["Percentage"])],
    textposition="outside",
    hovertemplate=
        "<b>%{x}</b><br>" +
        "Count: %{y}<br>" +
        "Percentage: %{customdata:.2f}%<extra></extra>",
    customdata=df_plot["Percentage"],
    marker=dict(
        color=["#d62728", "#1f77b4", "#2ca02c"],  # red, blue, green
        line=dict(color="black", width=1)
    )
))

# Layout customization
fig.update_layout(
    title={
        "text": "Class Distribution in Training Data",
        "x": 0.5,
        "xanchor": "center",
        "font": dict(size=20)
    },
    xaxis_title="Sentiment Class",
    yaxis_title="Number of Samples",
    template="plotly_white",
    hovermode="x unified",
    bargap=0.3,
    width=800,
    height=500
)

fig.show()

### Step 5: Clean and Prepare the Data

In [43]:
import nltk
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def clean_text(text):
    if pd.isna(text):   
        return ""
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = ' '.join(word for word in text.split() if word not in stop_words)
    return text

train['clean_text'] = train['safe_text'].apply(clean_text)
test['clean_text'] = test['safe_text'].apply(clean_text)

print(f"Train shape: {train.shape}, Test shape: {test.shape}")
print(train[['safe_text', 'clean_text']].head())

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\kinut\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Train shape: (10000, 5), Test shape: (5177, 3)
                                           safe_text  \
0  Me &amp; The Big Homie meanboy3000 #MEANBOY #M...   
1  I'm 100% thinking of devoting my career to pro...   
2  #whatcausesautism VACCINES, DO NOT VACCINATE Y...   
3  I mean if they immunize my kid with something ...   
4  Thanks to <user> Catch me performing at La Nui...   

                                          clean_text  
0  amp big homie meanboy meanboy mb mbs mmr stegm...  
1  im thinking devoting career proving autism isn...  
2          whatcausesautism vaccines vaccinate child  
3  mean immunize kid something wont secretly kill...  
4  thanks user catch performing la nuit nyc st av...  


In [44]:
# define target and features
X_train = train['clean_text']
y_train = train['label']
X_test = test['clean_text']

In [45]:

# train validation split
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=42)

### Step 6: Convert Text to Numerical Data

In [46]:
# vectorize text data
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_tf = vectorizer.fit_transform(X_train)
X_val_tf = vectorizer.transform(X_val)
X_test = vectorizer.transform(X_test)

In [48]:
## Handle class imbalance
# Logistic Regression and Random Forest we used class_weight='balanced'
# For KNN, we are oversampling using SMOTE
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_tf, y_train)

### Step 7: Train a Classification Model

In [49]:
# Defining evaluation function
def train_and_evaluate(model, X_train, y_train, X_val, y_val):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    
    print(f"Model: {model.__class__.__name__}")
    print(classification_report(y_val, y_pred, zero_division=0))
    
    acc = accuracy_score(y_val, y_pred)
    prec = precision_score(y_val, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_val, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_val, y_pred, average='weighted', zero_division=0)
    
    print(f"Accuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}, F1-Score: {f1:.4f}")
    print("-"*60)

In [50]:
# Fitting models

# Logistic Regression 
lr = LogisticRegression(max_iter=200, class_weight='balanced')
train_and_evaluate(lr, X_train_tf, y_train, X_val_tf, y_val)

# Random Forest 
rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
train_and_evaluate(rf, X_train_tf, y_train, X_val_tf, y_val)

# KNN 
knn = KNeighborsClassifier(n_neighbors=5)
train_and_evaluate(knn, X_train_res, y_train_res, X_val_tf, y_val)

Model: LogisticRegression
              precision    recall  f1-score   support

          -1       0.36      0.58      0.45       208
           0       0.82      0.76      0.79       982
           1       0.72      0.68      0.70       810

    accuracy                           0.71      2000
   macro avg       0.64      0.67      0.65      2000
weighted avg       0.73      0.71      0.72      2000

Accuracy: 0.7085, Precision: 0.7342, Recall: 0.7085, F1-Score: 0.7179
------------------------------------------------------------
Model: RandomForestClassifier
              precision    recall  f1-score   support

          -1       0.62      0.19      0.29       208
           0       0.80      0.80      0.80       982
           1       0.68      0.80      0.74       810

    accuracy                           0.74      2000
   macro avg       0.70      0.60      0.61      2000
weighted avg       0.73      0.74      0.72      2000

Accuracy: 0.7370, Precision: 0.7321, Recall: 0.7370

In [52]:
# Collecting metrics in a DataFrame for easier comparison
def get_metrics_only(model, X_val, y_val):
    """
    Calculates metrics from an already fitted model.
    """
    y_pred = model.predict(X_val)
    return {
        "Model": model.__class__.__name__,
        "Accuracy": accuracy_score(y_val, y_pred),
        "Precision": precision_score(y_val, y_pred, average='weighted', zero_division=0),
        "Recall": recall_score(y_val, y_pred, average='weighted', zero_division=0),
        "F1-Score": f1_score(y_val, y_pred, average='weighted', zero_division=0)
    }


metrics_df = pd.DataFrame([
    get_metrics_only(lr, X_val_tf, y_val),
    get_metrics_only(rf, X_val_tf, y_val),
    get_metrics_only(knn, X_val_tf, y_val)
])
metrics_df

,Model,Accuracy,Precision,Recall,F1-Score
0,LogisticRegression,0.7085,0.734171,0.7085,0.717949
1,RandomForestClassifier,0.7370,0.732140,0.7370,0.720662
2,KNeighborsClassifier,0.5090,0.700214,0.5090,0.550822


In [53]:
# Plot F1-score for comparison
import plotly.express as px

fig = px.bar(
    metrics_df,
    x="Model",
    y="F1-Score",
    text="F1-Score",
    hover_data={
        "Accuracy": True,
        "Precision": True,
        "Recall": True,
        "F1-Score": ":.4f"
    },
    color="Model",
    color_discrete_sequence=["#1f77b4", "#2ca02c", "#d62728"],
    title="F1-Score Comparison of Models"
)

fig.update_traces(texttemplate="%{text:.4f}", textposition="outside")
fig.update_layout(
    yaxis=dict(title="Weighted F1-Score", range=[0,1]),
    xaxis_title="Model",
    template="plotly_white",
    width=800,
    height=500
)

fig.show()

In [54]:
# Assuming you have metrics_df
best_model_name = metrics_df.loc[metrics_df["F1-Score"].idxmax(), "Model"]
best_f1 = metrics_df["F1-Score"].max()

print(f"Best model: {best_model_name} with F1-score: {best_f1:.4f}")

Best model: RandomForestClassifier with F1-score: 0.7207


In [58]:
#retrain best model on full training data

# Combine TF-IDF features
X_full_tf = vstack([X_train_tf, X_val_tf])
y_full = pd.concat([y_train, y_val])

# refit Random Forest on full data
rf_full = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight='balanced'
)

# Fit on full data
rf_full.fit(X_full_tf, y_full)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

### Step 8: Create Final Predictions

In [59]:
# Predict on test set
test_predictions = rf_full.predict(X_test)

# Prepare submission file
submission = pd.DataFrame({
    'tweet_id': test['tweet_id'],
    'sentiment': test_predictions
})

submission.to_csv("submission.csv", index=False)
print("Submission file created successfully!")

Submission file created successfully!
